# Getting your feet wet with geospatial data in Python

This notebook runs entirely in your browser: nothing to install, nothing sent to a server. Run a cell with **Shift + Enter**.

Your changes are kept in this browser only. To keep your work, use **File › Download**.

The notebook goes from single geometries, through a GeoJSON file, to a table of real fields at Ledreborg, and ends on a map.

In [ ]:
import json
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPoint

## 1. Points, lines and polygons

Any area of the Earth's surface, however small, holds infinitely many locations. Vector data gets round that with a few points: a line stands for every point between its ends, and a polygon for every point inside its outline.

In [ ]:
p1, p2, p3, p4, p5 = Point(1, 1), Point(1, 10), Point(10, 10), Point(10, 1), Point(5, 5)
MultiPoint([p1, p2, p3, p4, p5])

In [ ]:
line = LineString([p1, p3])
line

The line is defined by two points, but it stands for all the points on the straight line between them. `p5` was never used to build it, yet it lies on it:

In [ ]:
line.distance(p5)

In [ ]:
poly = Polygon([p1, p2, p3, p4])
poly

In [ ]:
print("distance from p5 to the polygon:", poly.distance(p5))
print("area:", poly.area, " perimeter:", poly.length)
print("does the polygon contain p5?", poly.contains(p5))

**Try:** make a point outside the polygon, e.g. `Point(15, 5)`, and ask for its distance to `poly`.

## 2. GeoJSON is just text

A GeoJSON file is text in a fixed structure: a list of *features*, each with a *geometry* (type and coordinates) and *properties* (the attributes). Read it as plain JSON first:

In [ ]:
with open("data/buildings.geojson") as f:
    data = json.load(f)

for feature in data["features"]:
    print(feature["properties"], feature["geometry"]["type"], len(feature["geometry"]["coordinates"]), "ring(s)")

GeoPandas reads the same file into a table with a geometry column, and knows its coordinate reference system (CRS):

In [ ]:
buildings = gpd.read_file("data/buildings.geojson")
print(buildings.crs)
buildings

In [ ]:
buildings["area_m2"] = buildings.area
buildings.plot(edgecolor="black")
buildings

The coordinates are in metres (ETRS89 / UTM zone 32N, EPSG:25832), so `area` gives square metres. One of the buildings has a hole, an inner courtyard: a polygon can have inner rings as well as an outline.

## 3. Real fields: Ledreborg 2026

The fields farmers declared for 2026 at Ledreborg, near Roskilde, with the crop and a simplified crop class. The file is in WGS 84 (longitude, latitude), as maps on the web expect.

In [ ]:
fields = gpd.read_file("data/ledreborg_fields_2026.geojson")
print(fields.crs, len(fields), "fields")
fields.head()

Areas in degrees mean nothing. Project to UTM 32N first, then measure:

In [ ]:
fields_utm = fields.to_crs(25832)
fields_utm["area_ha"] = fields_utm.area / 10000
fields_utm["area_ha"].sum()

In [ ]:
by_class = fields_utm.groupby("crop_class")["area_ha"].sum().sort_values()
by_class.plot.barh(figsize=(6, 5), title="Area per crop class, ha")
plt.show()

In [ ]:
fields_utm.plot(column="crop_class", legend=True, figsize=(9, 9),
                legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1)})
plt.show()

**Try:** select only the winter wheat fields with `fields_utm[fields_utm["crop_class"] == "Winter wheat"]` and plot them.

## 4. A simple overlay: fields per landscape

The study area is divided into five landscape types. An *intersection* cuts the fields along the landscape borders, so each piece knows its landscape:

In [ ]:
landscapes = gpd.read_file("data/ledreborg_landscapes.geojson").to_crs(25832)
pieces = gpd.overlay(fields_utm, landscapes, how="intersection")
pieces["area_ha"] = pieces.area / 10000
pieces.groupby(["landscape", "crop_class"])["area_ha"].sum().unstack(fill_value=0).round(1)

And a *buffer*: everything within 100 m of the valley system.

In [ ]:
valley = landscapes[landscapes["landscape"] == "Valley system"].dissolve()
near_valley = fields_utm[fields_utm.intersects(valley.buffer(100).iloc[0])]
ax = valley.plot(color="lightblue", figsize=(8, 8))
near_valley.plot(ax=ax, color="orange", edgecolor="black")
plt.show()
print(len(near_valley), "fields within 100 m of the valley system")

## 5. On a map

JupyterGIS draws the data on an interactive map with a basemap. `explore` opens one next to the notebook:

In [ ]:
from jupytergis import explore

explore(fields, layer_name="Fields 2026")

Or build a map yourself, layer by layer, and show it in the notebook:

In [ ]:
from jupytergis import GISDocument

doc = GISDocument()
doc.add_raster_layer("https://tile.openstreetmap.org/{z}/{x}/{y}.png", name="OpenStreetMap",
                     attribution="© OpenStreetMap contributors")
doc.add_geojson_layer(data=json.loads(landscapes.to_crs(4326).to_json()), name="Landscapes", opacity=0.5)
doc.add_geojson_layer(data=json.loads(near_valley.to_crs(4326).to_json()), name="Fields near the valley", zoom_to=True)
doc

## 6. Saving your results

Files you write appear in the file browser on the left; right-click one and choose **Download** to take it home.

A **GeoJSON** file can be written straight there:

In [ ]:
near_valley.to_file("fields_near_valley.geojson", driver="GeoJSON")

A **GeoPackage** is a small database, and the browser's file storage cannot be written to the way a database needs. Write it to `/tmp` first (the kernel's own memory), then copy the finished file across:

In [ ]:
import shutil

near_valley.to_file("/tmp/fields_near_valley.gpkg", driver="GPKG")
shutil.copy("/tmp/fields_near_valley.gpkg", "fields_near_valley.gpkg")
gpd.read_file("fields_near_valley.gpkg").head()

Reading a GeoPackage you have uploaded works directly; GDAL may warn that it opens it read-only, which you can ignore.

Everything here, your notebooks and files, is kept in this browser only. Download what you want to keep.

Data: Marker 2026 (Styrelsen for Grøn Arealomlægning og Vandmiljø), crop classes from SemanticGIS's [Danish crop codes](https://semanticgis.org/data/code-lists/danish-crop-codes-(afgkode)), landscapes digitised for the SemanticGIS Ledreborg examples. Map tiles © OpenStreetMap contributors.